# BOZZA - Test Classificatori: `trad_aug` vs `Real+Synth` sul Test Reale

> **Bozza di lavoro** (adattamento del `09_Test_Classificatori` del collega, che testa le 3 config
> Baseline/Real+Synth/Full Synth). Questo e' il notebook `11_Valutazione_Sostenibilita`: quando la
> bozza e' validata si esegue direttamente questo (copiandolo in `MammoDiffusion/notebooks/`).

Confronto sul **test set reale** (438 img: 365 sano / 73 malato) tra:

| Configurazione | Dati di training | Owner |
|---|---|---|
| **trad_aug** | reali + augmentation tradizionale (geometrica) | Samuele (rif. notebook 10) |
| **Real+Synth** | reali + immagini sintetiche (diffusione) | Enzo (rif. notebook 06) |

**Domanda ("D3" come la intende Samuele):** la generazione di sintetici migliora le prestazioni
abbastanza da giustificarne le emissioni? Qui si copre la **parte prestazionale** (tecnicamente D2);
la parte emissioni (kWh/CO2 via `eco_tracker`) e' in carico ai generatori.

**Protocollo:** stesso preprocessing del training (grayscale -> resize 224 -> /255 -> repeat3 -> norm
ImageNet); soglia operativa = punto di Youden **ricalcolato sul validation set** per entrambi i modelli
con la STESSA procedura (apples-to-apples, nessun leakage), poi congelata e applicata al test. NB: il
collega nel 09 carica le soglie gia' salvate (`val_metrics.json`); il 10 non salva quel file, quindi
qui le ricalcoliamo per entrambi (e si verifica che real_synth riproduca ~0.269 del collega).

**Run a 1 o 2 modelli (flag `MODELLI_ATTIVI` nella config):**
- `['trad_aug']` -> valida la pipeline dell'11 facendola girare col solo trad_aug: le metriche devono
  **riprodurre** il notebook 10 (AUC 0.5854, F1 0.2911, recall 0.6301, soglia 0.1198).
- `['trad_aug', 'Real+Synth']` -> confronto D3 a 2 vie. **OGGI (2026-06-19): SIMULAZIONE** con un `.keras`
  **VECCHIO** di real_synth (la versione attuale del collega, run `exp20260617`), recuperato e copiato a
  mano nell'exp dir locale. Serve a validare l'intera pipeline a 2 modelli prima che arrivi il modello
  riaddestrato. **DOMANI** il collega sovrascrive quel `.keras` col **riaddestrato** -> si rilancia per i
  numeri D3 **definitivi**.

**Stato bozza:** (1) `TRAD_AUG_MODEL_DRIVE_ID` serve SOLO se si gira fuori dal portatile (es. Colab); sul
portatile il `.keras` trad_aug e' gia' in `experiments/exp_trad_aug_resnet50/` e viene caricato in locale.
(2) Run **10** a batch 16 **FATTO** (2026-06-18) -> `.keras` trad_aug rigenerato. (3) `.keras` real_synth:
OGGI presente in locale (vecchio, posato a mano in `experiments/exp20260617_real_synth_resnet50_fine_tuned_batch_size_16/`
come `real_synth_resnet50_final_best.keras`). Col modello riaddestrato di domani aggiornare
`REAL_SYNTH_MODEL_DRIVE_ID` e `REAL_SYNTH_REF_THRESHOLD_YOUDEN`.

#### Verifica dell'ambiente di lavoro
Calcolo del percorso base in funzione dell'ambiente (locale/WSL o Google Colab).

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/MammoDiffusion/'
else:
    print("Ambiente locale/WSL rilevato.")
    BASE_PATH = '../'

print("Percorso base:", BASE_PATH)

#### Import librerie

In [ ]:
import zipfile
import gdown
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score,
    classification_report, precision_score, recall_score,
    f1_score, accuracy_score
)

# memory growth (utile sul portatile: carichiamo 2 ResNet-50)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print("set_memory_growth:", e)

#### Configurazione

In [ ]:
# --- Modelli a confronto -----------------------------------------------------
# trad_aug (Samuele): locale in experiments/, da caricare su Drive (poi mettere l'ID qui sotto)
TRAD_AUG_EXP_DIR        = os.path.join(BASE_PATH, 'experiments', 'exp_trad_aug_resnet50')
TRAD_AUG_MODEL_DRIVE_ID = ''   # TODO: inserire l'ID Drive dopo l'upload del .keras trad_aug

# real_synth (Enzo, notebook 06). Per la SIMULAZIONE D3 di OGGI (2026-06-19) si usa un .keras VECCHIO
# del collega = la sua versione real_synth ATTUALE (run exp20260617 / notebook 09: val AUC 0.6843,
# soglia Youden 0.2689), copiato a MANO nell'exp dir locale qui sotto. DOMANI il collega lo sovrascrive
# col modello RIADDESTRATO -> i numeri D3 DEFINITIVI verranno da quello. Il loader (b12) prende PRIMA il
# file locale nell'exp dir, poi (fallback) il Drive ID.
# IMPORTANTE: il file locale deve chiamarsi ESATTAMENTE 'real_synth_resnet50_final_best.keras' dentro
# REAL_SYNTH_EXP_DIR, altrimenti b12 ripiega sul download da Drive (controllare il print "[Real+Synth]
# carico dalla cartella esperimento").
REAL_SYNTH_EXP_DIR        = os.path.join(BASE_PATH, 'experiments', 'exp20260617_real_synth_resnet50_fine_tuned_batch_size_16')
REAL_SYNTH_MODEL_DRIVE_ID = '11f9pcL8kbLz7hi6i3lqDuferljdRl2uV'   # fallback se il .keras non e' nell'exp dir
# in alternativa alla soglia ricalcolata sotto, il collega salva la sua in val_metrics.json:
REAL_SYNTH_METRICS_DRIVE_ID = '1J55coTQluAadvUsusaR2uzP5juNCYnKx'
# soglia Youden del collega (val_metrics.json, exp20260617) usata SOLO come cross-check di quella che
# ricalcoliamo sul val: devono coincidere (~0.269) perche' il confronto sia apples-to-apples. Per il
# .keras VECCHIO di oggi questo valore e' corretto (val_metrics.json: 0.2689).
# ATTENZIONE: col modello RIADDESTRATO di domani sia REAL_SYNTH_MODEL_DRIVE_ID (o l'exp dir locale)
# sia questa soglia di riferimento vanno AGGIORNATI col nuovo val_metrics.json.
REAL_SYNTH_REF_THRESHOLD_YOUDEN = 0.269

# --- Modelli ATTIVI in questo run -------------------------------------------
# Permette di girare il notebook con UNO o DUE modelli senza toccare le altre celle:
#   ['trad_aug']               -> SOLO trad_aug. Valida la pipeline dell'11: le metriche trad_aug devono
#                                 RIPRODURRE quelle del notebook 10
#                                 (experiments/exp_trad_aug_resnet50/test_metrics_resnet50.json:
#                                 AUC 0.5854, F1 0.2911, recall 0.6301, soglia 0.1198).
#   ['trad_aug', 'Real+Synth'] -> confronto D3 a 2 vie. OGGI (2026-06-19) usato per la SIMULAZIONE col
#                                 .keras VECCHIO di real_synth (vedi sopra): valida l'intera pipeline a 2
#                                 modelli e dovrebbe riprodurre i riferimenti del collega (val AUC ~0.684,
#                                 soglia ~0.269, test AUC ~0.6122). DOMANI si rilancia col real_synth
#                                 RIADDESTRATO per i numeri D3 DEFINITIVI.
MODELLI_ATTIVI = ['trad_aug', 'Real+Synth']   # SIMULAZIONE D3 con real_synth VECCHIO locale (2026-06-19)

# Dataset preprocessato (zip su Drive come fallback)
PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
TEST_CSV_PATH = os.path.join(BASE_PATH, 'data', 'processed', 'metadata', 'test.csv')
VAL_CSV_PATH  = os.path.join(BASE_PATH, 'data', 'processed', 'metadata', 'val.csv')

# Preprocessing (identico al training, notebook 06/10)
IMG_SIZE = (224, 224)
BATCH_SIZE = 16   # se OOM sul portatile (qui si tengono fino a 2 ResNet-50 in memoria) abbassare a 8
SEED = 42
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406])
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225])
AUTOTUNE = tf.data.AUTOTUNE

# Output: cartella DEDICATA per non sovrascrivere results/test_classificatori dei colleghi
OUTPUT_DIR      = os.path.join(BASE_PATH, 'results', 'test_trad_aug_vs_real_synth')
FIGURES_DIR     = os.path.join(OUTPUT_DIR, 'figures')
TABLES_DIR      = os.path.join(OUTPUT_DIR, 'tables')
PREDICTIONS_DIR = os.path.join(OUTPUT_DIR, 'predictions')
MODELS_DIR      = os.path.join(OUTPUT_DIR, 'models')
for d in (OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, PREDICTIONS_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)

# validazione lista modelli attivi
_VALIDI = {'trad_aug', 'Real+Synth'}
assert 1 <= len(MODELLI_ATTIVI) and set(MODELLI_ATTIVI) <= _VALIDI, \
    "MODELLI_ATTIVI deve essere un sottoinsieme non vuoto di %s" % _VALIDI

print("Output in:", OUTPUT_DIR)
print("Modelli attivi in questo run:", MODELLI_ATTIVI)

#### Verifica presenza dataset preprocessato

In [ ]:
PROCESSED_ZIP = os.path.join(BASE_PATH, 'data', 'processed', 'processed.zip')

def verifica_processed_data():
    if not (os.path.isfile(TEST_CSV_PATH) and os.path.isfile(VAL_CSV_PATH)):
        return False
    try:
        first_path = pd.read_csv(TEST_CSV_PATH)['processed_path'].iloc[0]
        return os.path.isfile(os.path.join(BASE_PATH, first_path))
    except Exception:
        return False

if not verifica_processed_data():
    print("Dataset non trovato, scarico da Google Drive...")
    os.makedirs(os.path.dirname(PROCESSED_ZIP), exist_ok=True)
    gdown.download(id=PROCESSED_DRIVE_ID, output=PROCESSED_ZIP, quiet=False)
    with zipfile.ZipFile(PROCESSED_ZIP, 'r') as z:
        z.extractall(os.path.join(BASE_PATH, 'data', 'processed'))
    os.remove(PROCESSED_ZIP)
    print("Dataset estratto.")
else:
    print("Dataset gia presente, salto il download.")

#### Creazione dataset Validation e Test
Solo immagini reali, preprocessing identico al training. Il VAL serve per la soglia di Youden, il
TEST per la valutazione finale.

In [ ]:
def caricamento_preprocessing(percorso, etichetta):
    raw = tf.io.read_file(percorso)
    img = tf.image.decode_image(raw, channels=1, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.repeat(img, repeats=3, axis=-1)
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return img, tf.cast(etichetta, tf.float32)

def crea_dataset(df):
    percorsi  = [os.path.join(BASE_PATH, p) for p in df['processed_path'].values]
    etichette = df['cancer'].values
    ds = tf.data.Dataset.from_tensor_slices((percorsi, etichette))
    ds = ds.map(lambda p, l: caricamento_preprocessing(p, l), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

df_val  = pd.read_csv(VAL_CSV_PATH).reset_index(drop=True)
df_test = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

print("VAL :", len(df_val),  "img  (malato=%d, sano=%d)" % ((df_val['cancer']==1).sum(),  (df_val['cancer']==0).sum()))
print("TEST:", len(df_test), "img  (malato=%d, sano=%d)" % ((df_test['cancer']==1).sum(), (df_test['cancer']==0).sum()))

val_ds  = crea_dataset(df_val)
test_ds = crea_dataset(df_test)
print("Dataset val/test pronti.")

#### Caricamento modelli
Priorita': (1) cartella esperimento locale, (2) cache `results/.../models/`, (3) download da Drive.
Caricati con `compile=False` (serve solo l'inferenza; evita di deserializzare focal loss/metriche).

In [ ]:
MODELS_CONFIG = {
    'trad_aug': {
        'percorso_exp':     os.path.join(TRAD_AUG_EXP_DIR, 'trad_aug_resnet50_final_best.keras'),
        'percorso_modello': os.path.join(MODELS_DIR,       'trad_aug_resnet50_final_best.keras'),
        'drive_id':         TRAD_AUG_MODEL_DRIVE_ID,
    },
    'Real+Synth': {
        'percorso_exp':     os.path.join(REAL_SYNTH_EXP_DIR, 'real_synth_resnet50_final_best.keras'),
        'percorso_modello': os.path.join(MODELS_DIR,         'real_synth_resnet50_final_best.keras'),
        'drive_id':         REAL_SYNTH_MODEL_DRIVE_ID,
    },
}

def carica_modello(nome, config):
    if os.path.isfile(config['percorso_exp']):
        print("[%s] carico dalla cartella esperimento" % nome)
        return keras.models.load_model(config['percorso_exp'], compile=False)
    if os.path.isfile(config['percorso_modello']):
        print("[%s] carico dalla cache models/" % nome)
        return keras.models.load_model(config['percorso_modello'], compile=False)
    if not config['drive_id']:
        raise FileNotFoundError(
            "[%s] modello assente in locale e drive_id mancante: inserire l'ID Drive "
            "o copiare il .keras nell'exp dir." % nome)
    print("[%s] scarico da Google Drive..." % nome)
    gdown.download(id=config['drive_id'], output=config['percorso_modello'], quiet=False)
    return keras.models.load_model(config['percorso_modello'], compile=False)

# carica SOLO i modelli attivi (vedi MODELLI_ATTIVI in b06). Per real_synth, in SIMULAZIONE oggi, deve
# stampare "carico dalla cartella esperimento" (= il .keras VECCHIO locale), NON "scarico da Google Drive".
modelli = {nome: carica_modello(nome, MODELS_CONFIG[nome]) for nome in MODELLI_ATTIVI}
print("\nModelli caricati (compile=False, solo inferenza):", list(modelli))

#### Soglie operative - punto di Youden dal Validation Set
Per entrambi i modelli la soglia e' il punto di Youden (J = TPR - FPR) calcolato **sul validation set**
con lo stesso preprocessing, poi congelata e applicata al test (nessun leakage). Procedura identica per
i due modelli -> confronto apples-to-apples. Per real_synth questo dovrebbe riprodurre la soglia ~0.269
che il collega aveva salvato in `val_metrics.json` (cross-check; in alternativa caricarla da Drive con
`REAL_SYNTH_METRICS_DRIVE_ID`).

In [ ]:
def predici(modello, dataset):
    y_true, y_prob = [], []
    for imgs, labels in dataset:
        y_prob.extend(modello.predict(imgs, verbose=0).flatten())
        y_true.extend(labels.numpy())
    return np.array(y_true), np.array(y_prob)

def soglia_youden(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])

# predizioni sul VAL per ricavare le soglie (val_ds senza shuffle -> ordinamento coerente fra modelli)
val_true  = None
val_probs = {}
soglie    = {}
for nome in MODELLI_ATTIVI:
    yt, yp = predici(modelli[nome], val_ds)
    if val_true is None:
        val_true = yt
    else:
        assert np.array_equal(val_true, yt), "ordine y_true del VAL incoerente fra modelli"
    val_probs[nome] = yp
    soglie[nome]    = soglia_youden(yt, yp)

print("Soglie di Youden (dal VAL):")
for nome in MODELLI_ATTIVI:
    print("  %-12s: %.4f" % (nome, soglie[nome]))

# cross-check apples-to-apples: la soglia real_synth ricalcolata deve riprodurre quella che il collega
# aveva salvato in val_metrics.json (notebook 06). Se diverge troppo, le metriche threshold-dependent
# (F1/recall/precision/accuracy) di real_synth non combaceranno con la sua tabella del 09 -> da indagare
# (val set diverso? modello diverso?). AUC e' invece threshold-independent -> non ne risente.
if 'Real+Synth' in MODELLI_ATTIVI:
    _delta_thr = abs(soglie['Real+Synth'] - REAL_SYNTH_REF_THRESHOLD_YOUDEN)
    print("\nCross-check soglia real_synth: ricalcolata %.4f vs riferimento collega %.4f (delta %.4f)" % (
        soglie['Real+Synth'], REAL_SYNTH_REF_THRESHOLD_YOUDEN, _delta_thr))
    if _delta_thr > 0.02:
        print("  ATTENZIONE: scostamento > 0.02 -> verificare val/modello prima di confrontare col 09.")
    else:
        print("  OK: coerente col collega -> confronto apples-to-apples valido.")
else:
    print("\n(Real+Synth non attivo: cross-check soglia ~0.269 rimandato al run a due modelli.)")

#### Predizioni sul Test Set

In [ ]:
# predizioni sul TEST per ogni modello attivo (test_ds senza shuffle -> allineato a df_test)
test_true  = None
test_probs = {}
for nome in MODELLI_ATTIVI:
    yt, yp = predici(modelli[nome], test_ds)
    if test_true is None:
        test_true = yt
    else:
        assert np.array_equal(test_true, yt), "ordine y_true del TEST incoerente fra modelli"
    test_probs[nome] = yp
print("Predizioni sul test set completate per:", list(test_probs))

In [ ]:
# nomi colonna compatti per ogni config: 'trad_aug' -> trad_aug, 'Real+Synth' -> real_synth
def _key(nome):
    return nome.lower().replace('+', '_')

df_pred = df_test[['processed_path', 'cancer']].copy().rename(columns={'cancer': 'label_true'})
for nome in MODELLI_ATTIVI:
    k = _key(nome)
    df_pred['prob_' + k] = test_probs[nome]
    df_pred['pred_' + k] = (test_probs[nome] >= soglie[nome]).astype(int)

preds_path = os.path.join(PREDICTIONS_DIR, 'test_predictions.csv')
df_pred.to_csv(preds_path, index=False)
print("Predizioni salvate in:", preds_path)
print(df_pred.head())

#### Valutazione - Metriche per Configurazione

In [ ]:
CLASSIFICATORI = {nome: (test_probs[nome], soglie[nome]) for nome in MODELLI_ATTIVI}

all_metrics = {}
for nome, (probs, thr) in CLASSIFICATORI.items():
    fpr, tpr, _ = roc_curve(test_true, probs)
    y_pred = (probs >= thr).astype(int)
    all_metrics[nome] = {
        'auc':       round(float(roc_auc_score(test_true, probs)), 4),
        'threshold': round(float(thr), 4),
        'accuracy':  round(float(accuracy_score(test_true, y_pred)), 4),
        'precision': round(float(precision_score(test_true, y_pred, pos_label=1, zero_division=0)), 4),
        'recall':    round(float(recall_score(test_true, y_pred, pos_label=1, zero_division=0)), 4),
        'f1':        round(float(f1_score(test_true, y_pred, pos_label=1, zero_division=0)), 4),
        'fpr': fpr, 'tpr': tpr, 'y_pred': y_pred,
    }
    print("\n" + "="*48)
    print(" ", nome)
    print("="*48)
    print("  AUC: %.4f | F1: %.4f | Recall: %.4f | Acc: %.4f | soglia: %.3f" % (
        all_metrics[nome]['auc'], all_metrics[nome]['f1'],
        all_metrics[nome]['recall'], all_metrics[nome]['accuracy'], all_metrics[nome]['threshold']))
    print(classification_report(test_true, y_pred, target_names=['Sano', 'Malato']))

# delta Real+Synth - trad_aug (per la "D3" di Samuele: quanto "compra" la generazione).
# Calcolabile solo col run a DUE modelli.
if 'trad_aug' in all_metrics and 'Real+Synth' in all_metrics:
    delta = {k: round(all_metrics['Real+Synth'][k] - all_metrics['trad_aug'][k], 4)
             for k in ('auc', 'f1', 'recall', 'accuracy')}
    print("\nDelta (Real+Synth - trad_aug):", delta)
else:
    delta = {}
    print("\nDelta non calcolato: run a modello singolo (%s)." % MODELLI_ATTIVI)

metrics_to_save = {
    'modelli_attivi': MODELLI_ATTIVI,
    'configs': {k: {m: v for m, v in vals.items() if m not in ('fpr', 'tpr', 'y_pred')}
                for k, vals in all_metrics.items()},
    'delta_real_synth_minus_trad_aug': delta,
}
out_json = os.path.join(TABLES_DIR, 'test_metrics_trad_aug_vs_real_synth.json')
with open(out_json, 'w') as f:
    json.dump(metrics_to_save, f, indent=2, ensure_ascii=False)
print("\nMetriche salvate in:", out_json)

#### Visualizzazione - Curve ROC a confronto

In [ ]:
COLORS = {'trad_aug': 'tab:orange', 'Real+Synth': 'tab:green'}
fig, ax = plt.subplots(figsize=(8, 6))
for nome, m in all_metrics.items():
    ax.plot(m['fpr'], m['tpr'], color=COLORS[nome], lw=2,
            label="%s  (AUC = %.4f)" % (nome, m['auc']))
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC - trad_aug vs Real+Synth (test reale)')
ax.legend(loc='lower right'); ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
roc_path = os.path.join(FIGURES_DIR, 'roc_trad_aug_vs_real_synth.png')
fig.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print("Salvato in:", roc_path)

#### Visualizzazione - Matrici di confusione

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nome, m) in zip(axes, all_metrics.items()):
    cm = confusion_matrix(test_true, m['y_pred'])
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Sano', 'Malato']); ax.set_yticklabels(['Sano', 'Malato'])
    ax.set_xlabel('Predetto'); ax.set_ylabel('Reale')
    ax.set_title("%s\nAUC=%.4f  F1=%.4f  Recall=%.4f" % (nome, m['auc'], m['f1'], m['recall']))
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=15, fontweight='bold', color=color)
plt.suptitle('Confusion Matrix - Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(FIGURES_DIR, 'cm_trad_aug_vs_real_synth.png')
fig.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print("Salvato in:", cm_path)

#### Visualizzazione - Confronto metriche (barre)

In [ ]:
metric_keys   = ['auc', 'accuracy', 'precision', 'recall', 'f1']
metric_labels = ['AUC', 'Accuracy', 'Precision\n(malato)', 'Recall\n(malato)', 'F1\n(malato)']
x = np.arange(len(metric_keys)); width = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
nomi = list(all_metrics.keys())   # itera sui modelli EFFETTIVI (1 o 2), non su COLORS
for i, nome in enumerate(nomi):
    color = COLORS.get(nome, 'tab:blue')
    vals = [all_metrics[nome][k] for k in metric_keys]
    bars = ax.bar(x + i * width, vals, width, label=nome, color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005, "%.3f" % v,
                ha='center', va='bottom', fontsize=8, fontweight='bold')
# centra le tick sotto il gruppo di barre (vale per 1 o 2 modelli)
ax.set_xticks(x + width * (len(nomi) - 1) / 2); ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Confronto metriche - trad_aug vs Real+Synth')
ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
bar_path = os.path.join(FIGURES_DIR, 'metrics_comparison.png')
fig.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print("Salvato in:", bar_path)

#### Tabella riassuntiva

In [ ]:
df_summary = pd.DataFrame([
    {'Configurazione': nome, 'AUC': "%.4f" % m['auc'], 'Accuracy': "%.4f" % m['accuracy'],
     'Precision': "%.4f" % m['precision'], 'Recall': "%.4f" % m['recall'],
     'F1': "%.4f" % m['f1'], 'Soglia': "%.3f" % m['threshold']}
    for nome, m in all_metrics.items()
]).set_index('Configurazione')
print(df_summary.to_string())
csv_path = os.path.join(TABLES_DIR, 'summary_trad_aug_vs_real_synth.csv')
df_summary.to_csv(csv_path)
print("\nTabella salvata in:", csv_path)

#### Robustezza statistica - bootstrap bilanciato
Con solo 73 positivi a test le metriche sono fragili. Si stima l'incertezza con un **bootstrap
bilanciato**: a ogni round si ricampiona **con reinserimento** un numero uguale di positivi e negativi
(= classe minoritaria). Se gli intervalli di trad_aug e Real+Synth si sovrappongono, il lift va
dichiarato **reale ma non robusto** (atteso con questa numerosita').

In [ ]:
def bootstrap_bilanciato(y_true, y_prob, threshold, n_rounds=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    n = min(len(pos_idx), len(neg_idx))
    out = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': [], 'ROC_AUC': []}
    for _ in range(n_rounds):
        sel_pos = rng.choice(pos_idx, size=n, replace=True)
        sel_neg = rng.choice(neg_idx, size=n, replace=True)
        idx = np.concatenate([sel_pos, sel_neg])
        yt, yp = y_true[idx], y_prob[idx]
        pred = (yp >= threshold).astype(int)
        out['Accuracy'].append(accuracy_score(yt, pred))
        out['Precision'].append(precision_score(yt, pred, pos_label=1, zero_division=0))
        out['Recall'].append(recall_score(yt, pred, pos_label=1, zero_division=0))
        out['F1'].append(f1_score(yt, pred, pos_label=1, zero_division=0))
        out['ROC_AUC'].append(roc_auc_score(yt, yp))
    return out

boot = {}
for nome, (probs, thr) in CLASSIFICATORI.items():
    boot[nome] = bootstrap_bilanciato(test_true, probs, thr)
    print("\n[%s]" % nome)
    for k, v in boot[nome].items():
        print("  %-10s %.4f +/- %.4f" % (k, np.mean(v), np.std(v)))

# boxplot AUC e F1 a confronto fra le due config
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ['ROC_AUC', 'F1']):
    ax.boxplot([boot[n][metric] for n in CLASSIFICATORI])
    ax.set_xticklabels(list(CLASSIFICATORI.keys()))
    ax.set_title('%s - bootstrap (test)' % metric)
    ax.set_ylabel('valore'); ax.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
boot_path = os.path.join(FIGURES_DIR, 'bootstrap_comparison.png')
fig.savefig(boot_path, dpi=150, bbox_inches='tight')
plt.show()
print("\nSalvato in:", boot_path)

#### Grad-CAM - Confronto attivazioni sul Test Set
Regioni attivate dai due classificatori sugli stessi 4 campioni (2 malato + 2 sano).

In [ ]:
IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406])
IMAGENET_STD_NP  = np.array([0.229, 0.224, 0.225])

def make_gradcam_heatmap(img_array, model):
    backbone = model.get_layer('resnet50')
    head_layers = [l for l in model.layers if l.name not in (model.layers[0].name, backbone.name)]
    with tf.GradientTape() as tape:
        backbone_out = backbone(img_array, training=False)
        tape.watch(backbone_out)
        x = backbone_out
        for layer in head_layers:
            x = layer(x, training=False)
        predictions = x
    grads = tape.gradient(predictions, backbone_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = backbone_out[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam(img_tensor, heatmap, ax, title=""):
    img_vis = img_tensor.numpy() * IMAGENET_STD_NP + IMAGENET_MEAN_NP
    img_vis = np.clip(img_vis, 0, 1)[:, :, 0]
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], IMG_SIZE).numpy().squeeze()
    ax.imshow(img_vis, cmap='gray')
    ax.imshow(heatmap_resized, cmap='jet', alpha=0.4)
    ax.set_title(title, fontsize=8); ax.axis('off')

sample_pos, sample_neg = [], []
for imgs, etichette in test_ds:
    for i in range(len(imgs)):
        et = int(etichette[i].numpy())
        if et == 1 and len(sample_pos) < 2:
            sample_pos.append((imgs[i], et))
        elif et == 0 and len(sample_neg) < 2:
            sample_neg.append((imgs[i], et))
    if len(sample_pos) >= 2 and len(sample_neg) >= 2:
        break
samples = sample_pos + sample_neg

# una colonna per modello attivo (1 oggi, 2 domani)
classifiers_list = [(nome, modelli[nome], soglie[nome]) for nome in MODELLI_ATTIVI]

# squeeze=False -> axes sempre 2D anche con un solo modello (evita IndexError su axes[row, col])
fig, axes = plt.subplots(len(samples), len(classifiers_list),
                         figsize=(4.5 * len(classifiers_list), 4 * len(samples)), squeeze=False)
for row, (img, et) in enumerate(samples):
    for col, (clf_name, clf_model, clf_thr) in enumerate(classifiers_list):
        img_batch = tf.expand_dims(img, 0)
        prob = float(clf_model.predict(img_batch, verbose=0)[0][0])
        pred = 'Malato' if prob >= clf_thr else 'Sano'
        heatmap = make_gradcam_heatmap(img_batch, clf_model)
        show_gradcam(img, heatmap, axes[row, col],
                     title="%s | reale: %s -> %s (%.2f)" % (clf_name, 'Malato' if et==1 else 'Sano', pred, prob))
plt.suptitle("Grad-CAM - trad_aug vs Real+Synth", fontsize=12, fontweight='bold')
plt.tight_layout()
gradcam_path = os.path.join(FIGURES_DIR, 'gradcam_comparison.png')
fig.savefig(gradcam_path, dpi=150, bbox_inches='tight')
plt.show()
print("Salvato in:", gradcam_path)

#### Lettura per il report (D2/D3 di Samuele)
> **OGGI (2026-06-19) e' una SIMULAZIONE:** il run a 2 vie gira col `.keras` **VECCHIO** di real_synth
> (versione attuale del collega, `exp20260617`). Serve a validare la pipeline e a fissare la narrativa,
> ma i numeri D3 **definitivi** arrivano col modello **riaddestrato** (domani) -> rieseguire allora.
> Atteso da questo run-simulazione: real_synth ~= riferimenti del collega (val AUC ~0.684, soglia ~0.269,
> test AUC ~0.6122); se combaciano, la pipeline a 2 modelli e' validata.

- Confrontare AUC e F1(malato): se **Real+Synth > trad_aug** -> la generazione aggiunge varieta' utile
  che l'augmentation geometrica (1020 copie degli stessi 340 positivi) non da'.
- **Narrativa aggiornata al run FINALE batch 16 (NON piu' batch 8):** il margine di Real+Synth su
  trad_aug e' sceso a **+0.027 AUC** (era +0.066), **dentro la std del bootstrap (~+/-0.046)** -> il
  vantaggio della generazione e' **reale ma NON robusto**: con 73 positivi a test NON va dichiarato
  significativo. trad_aug, inoltre, col batch 16 **supera la baseline real_only** (non e' piu' l'ultimo).
  NB: il +0.027 viene dal run precedente del collega; col real_synth **riaddestrato** potrebbe cambiare.
- Riportare l'**AUC** (threshold-independent) come metrica primaria: le soglie di Youden(val) sono
  basse e instabili (trad_aug ~0.12) -> le metriche threshold-dependent (F1/recall) sono fragili.
- Incrociare col **bootstrap**: gli intervalli di trad_aug e Real+Synth si **sovrappongono** -> lift
  **reale ma non robusto** (atteso con questa numerosita').
- **Emissioni (D3 vera)**: i kWh/CO2 di augmentation vs generazione li forniscono i generatori
  (`eco_tracker`); qui si quantifica solo il guadagno prestazionale che quelle emissioni "comprano".

**Checklist bozza prima dell'esecuzione finale (questo notebook = 11):**
1. `.keras` trad_aug presente in `experiments/exp_trad_aug_resnet50/` (sul portatile c'e' gia' dopo il
   run del 10, FATTO il 2026-06-18); `TRAD_AUG_MODEL_DRIVE_ID` serve SOLO se si gira su Colab/altra macchina.
2. **`MODELLI_ATTIVI` (cella config):** `['trad_aug']` = solo trad_aug (valida che l'11 riproduca il
   notebook 10). `['trad_aug', 'Real+Synth']` = confronto D3 a 2 vie. **OGGI = 2 vie in SIMULAZIONE** col
   `.keras` real_synth **VECCHIO** posato a mano nell'exp dir; **DOMANI** stesso flag col modello
   **riaddestrato** per i numeri definitivi.
3. **`.keras` real_synth nell'exp dir:** il file deve chiamarsi ESATTAMENTE
   `real_synth_resnet50_final_best.keras` in `experiments/exp20260617_real_synth_resnet50_fine_tuned_batch_size_16/`,
   altrimenti b12 ripiega su Drive. Col modello **RIADDESTRATO** aggiornare `REAL_SYNTH_MODEL_DRIVE_ID`
   (o sovrascrivere il `.keras` nell'exp dir) **e** `REAL_SYNTH_REF_THRESHOLD_YOUDEN` con la nuova soglia del val.
4. Cross-check soglia (cella b14): real_synth ricalcolata ~0.269 (= `val_metrics.json` del collega,
   0.2689) -> confronto apples-to-apples valido. Se diverge, la cella lo segnala.
5. Output in `results/test_trad_aug_vs_real_synth/` (non sovrascrive i risultati dei colleghi).